<a href="https://colab.research.google.com/github/erikh-1126/04-Project-Group/blob/main/04_manual_mlps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 LAB: Manual MLPs for Classification and Regression

In this lab, you will use `PyTorch` to implement manually a multi-layer perceptron (MLP) for three different tasks: binary classification, multi-class classification and regression.

## General instructions to complete in ALL three tasks:

1. ***IMPLEMENTATION***:
   
Implement a separate class for each task:

  - `BinaryMLP` for the binary classification task  
  - `MultiClassMLP` for the multi-class classification task  
  - `RegressionMLP` for the regression task

   Each class must include the following methods:

  - `__init__` for initializing 1 or 2 hidden layers. (*Help*: having a second hidden layer only means the need of having a extra set of weights and intercept).
  - `forward` to transfer information from the input to the output layer. (*Help*: make sure that information passes through a second hidden layer if there is one).
  - `cost` computing the cost.
  - `fit` for training, using the gradients to manually update the parameters of the model. **Use stochastic gradient descent to update your weights**. *N.B.* You may probably reuse much of the code we used in this week tutorial already.
  - `predict` to convert the information at the output layer into the required output.

2. ***DATA PREPARATION***

For each task, you will be provided with a toy dataset. For each dataset:

  - Split into training and holdout sets (use an 80/20 split)
  - Standardize the features properly, avoiding data leakage
  - Convert all data into `PyTorch` Tensors for compatibility

3. ***MODEL TRAINING AND EVALUATION***

Instantiate the model for each task and train it under different hyperparameter configurations. In each case, record the performance on **both** the training and holdout sets using the appropriate metric for the task (accuracy, MSE, etc.). You should explore the following configurations:

  - One hidden layer, varying the number of hidden units (use ReLU as their activation function)
  - Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
  - Repeat the above setups using Tanh activation instead of ReLU

Present your results in a compact way (e.g., a summary table, a data frame etc).

**NOTE**: When training your model, use a fixed learning rate of your choice (e.g., 0.01 is a reasonable starting point) and a reasonably large number of  epochs (e.g., 100–200) based on how training and holdout performance evolve.

4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?
- What changes when using two layers instead of one?
- How does the activation function (ReLU vs. Tanh) influence results?

Please elaborate your answers.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

**Ideally, each group member should be responsible for one of the separate tasks.** BUT, everyone should help each other along the way, both reviewing and refining results and discussion.

---

In total, this lab assignment will be worth **100 points**.

---
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


In [ ]:
NAME(s) = "Erik H., Derek W., Henry Y."

SyntaxError: cannot assign to function call here. Maybe you meant '==' instead of '='? (147228175.py, line 1)

## 1. Pre-implementation Group Discussion (15 points)

Discuss and agree on:

- What cost function should be used for each of the below task.
- What changes are needed in the output layer for each of these tasks. In particular, consider the number of units and the activation function.
- Why it is important to standardize the data before training each model.
- How you could detect overfitting when training your models.

For a binary MLP, a Binary Cross-Entropy Cost Function is needed to perform it. For a multi-class MLP, a Categorical Cross-Entropy is needed instead. Meanwhile, linear regression MLPs need a MSE function as its cost function.

For both binary and linear regression MLPs, one output neuron is needed, whereas the number of neurons needed for a multiclass MLP depends on the number of classes. In terms of the activation function, a binary MLP requires a Sigmoid function, a multiclass MLP requires a Softmax function, and a linear regression MLP doesn't even require a specific activation function.

When we standardize a function, each of the features will have an equal level of input into the function, so it prevents functions with disparagingly high values from dominating the learning process of the model.

In order to prevent overfitting, if the dataset isn't too large, cross-validation can be used as a remedy, as it exposes the model to unseen data. If the data adjusts well to it, then it should be fine. Otherwise, there may be some overfitting involved.

## 2. Binary Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [64]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

In [ ]:
# USE AS MANY CELLS AS NEEDED

In [65]:
import torch


def sigmoid(z):
    return 1 / (1 + torch.exp(-z))


def relu(z):
    return torch.maximum(z, torch.tensor(0.0))


class BinaryMLP:

    def __init__(
        self,
        input_dim,
        hidden_dim,
        hidden_dim2=None,
        learning_rate=0.1,
        activation="relu"
    ):
        # Input layer -> first hidden layer
        self.W_1 = torch.randn(input_dim, hidden_dim) * 0.1
        self.b_1 = torch.zeros(hidden_dim)

        # Optional second hidden layer
        self.hidden_dim2 = hidden_dim2
        self.activation = activation

        if hidden_dim2 is not None:
            self.W_2 = torch.randn(hidden_dim, hidden_dim2) * 0.1
            self.b_2 = torch.zeros(hidden_dim2)
            output_input_dim = hidden_dim2
        else:
            output_input_dim = hidden_dim

        # Final hidden layer -> output layer
        self.W_out = torch.randn(output_input_dim, 1) * 0.1
        self.b_out = torch.zeros(1)

        self.learning_rate = learning_rate

    def forward(self, X):
        self.h_1 = X @ self.W_1 + self.b_1

        if self.activation == "relu":
            self.a_1 = torch.relu(self.h_1)
        else:
            self.a_1 = torch.tanh(self.h_1)

        if self.hidden_dim2 is not None:
            self.h_2 = self.a_1 @ self.W_2 + self.b_2

            if self.activation == "relu":
                self.a_2 = torch.relu(self.h_2)
            else:
                self.a_2 = torch.tanh(self.h_2)

            hidden_output = self.a_2
        else:
            hidden_output = self.a_1

        output = hidden_output @ self.W_out + self.b_out
        return sigmoid(output)

    def cost(self, y_pred, y_true):
        # Binary cross-entropy
        eps = torch.finfo(y_pred.dtype).eps
        y_pred = torch.clamp(y_pred, eps, 1 - eps)

        return -torch.mean(
            y_true * torch.log(y_pred)
            + (1 - y_true) * torch.log(1 - y_pred)
        )

    def fit(self, X, y, n_epochs=100):
        X = torch.as_tensor(X, dtype=torch.float32)
        y = torch.as_tensor(y, dtype=torch.float32).reshape(-1, 1)

        n = X.shape[0]
        self.loss_history = []

        for epoch in range(n_epochs):
            # Stochastic gradient descent: shuffle observations
            randperm = torch.randperm(n)

            for ii in randperm:
                x_batch = X[ii:ii + 1]
                y_batch = y[ii:ii + 1]

                # Forward pass
                y_pred = self.forward(x_batch)

                # Output gradient: sigmoid + binary cross-entropy
                delta_out = y_pred - y_batch

                # Gradient for output layer
                if self.hidden_dim2 is not None:
                    dW_out = self.a_2.T @ delta_out
                else:
                    dW_out = self.a_1.T @ delta_out

                db_out = delta_out.sum(dim=0)

                # Backpropagate through optional second hidden layer
                if self.hidden_dim2 is not None:
                    if self.activation == "relu":
                        derivative_2 = (self.h_2 > 0).float()
                    else:
                        derivative_2 = 1 - self.a_2 ** 2

                    delta_2 = (delta_out @ self.W_out.T) * derivative_2

                    dW_2 = self.a_1.T @ delta_2
                    db_2 = delta_2.sum(dim=0)

                    if self.activation == "relu":
                        derivative_1 = (self.h_1 > 0).float()
                    else:
                        derivative_1 = 1 - self.a_1 ** 2

                    delta_1 = (delta_2 @ self.W_2.T) * derivative_1

                else:
                    if self.activation == "relu":
                        derivative_1 = (self.h_1 > 0).float()
                    else:
                        derivative_1 = 1 - self.a_1 ** 2

                    delta_1 = (delta_out @ self.W_out.T) * derivative_1

                dW_1 = x_batch.T @ delta_1
                db_1 = delta_1.sum(dim=0)

                # Manual gradient descent updates
                with torch.no_grad():
                    self.W_1 -= self.learning_rate * dW_1
                    self.b_1 -= self.learning_rate * db_1

                    self.W_out -= self.learning_rate * dW_out
                    self.b_out -= self.learning_rate * db_out

                    if self.hidden_dim2 is not None:
                        self.W_2 -= self.learning_rate * dW_2
                        self.b_2 -= self.learning_rate * db_2

            # Record loss at the end of each epoch
            with torch.no_grad():
                predictions = self.forward(X)
                epoch_loss = self.cost(predictions, y)
                self.loss_history.append(epoch_loss.item())

            if epoch % 10 == 0:
                print(
                    f"Epoch {epoch}, "
                    f"Cost: {self.loss_history[-1]:.4f}"
                )

        return self

    def predict(self, X):
        X = torch.as_tensor(X, dtype=torch.float32)

        with torch.no_grad():
            probabilities = self.forward(X)
            predicted = (probabilities >= 0.5).float()

        return predicted

In [66]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

# 1. Split into training and holdout sets (80/20)
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# 2. Standardize features without data leakage
scaler = StandardScaler()

# Fit ONLY on the training features
X_train = scaler.fit_transform(X_train)

# Apply the same scaling to the holdout features
X_holdout = scaler.transform(X_holdout)

# 3. Convert features and labels into PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_holdout = torch.tensor(X_holdout, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_holdout = torch.tensor(y_holdout, dtype=torch.float32).reshape(-1, 1)

# Check the shapes
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_holdout:", X_holdout.shape)
print("y_holdout:", y_holdout.shape)

X_train: torch.Size([800, 2])
y_train: torch.Size([800, 1])
X_holdout: torch.Size([200, 2])
y_holdout: torch.Size([200, 1])


In [67]:
import pandas as pd
from sklearn.metrics import accuracy_score

# Fixed training settings
learning_rate = 0.01
epochs = 150

# Test different network sizes and activation functions
configs = []

# One hidden layer: vary the number of hidden units
for units in [8, 16, 32]:
    for activation in ["relu", "tanh"]:
        configs.append({
            "Architecture": f"1 hidden layer ({units})",
            "Hidden 1": units,
            "Hidden 2": None,
            "Activation": activation
        })

# Two hidden layers: use the same number of units in each layer
for units in [8, 16, 32]:
    for activation in ["relu", "tanh"]:
        configs.append({
            "Architecture": f"2 hidden layers ({units}, {units})",
            "Hidden 1": units,
            "Hidden 2": units,
            "Activation": activation
        })

results = []

for config in configs:
    # Create a fresh model for each experiment
    model = BinaryMLP(
        input_dim=X_train.shape[1],
        hidden_dim=config["Hidden 1"],
        hidden_dim2=config["Hidden 2"],
        learning_rate=learning_rate,
        activation=config["Activation"]
    )

    # Train the model
    model.fit(X_train, y_train, n_epochs=epochs)

    # Predict on both datasets
    train_pred = model.predict(X_train).detach().cpu().numpy().ravel()
    holdout_pred = model.predict(X_holdout).detach().cpu().numpy().ravel()

    # Convert true labels to NumPy arrays
    train_true = y_train.detach().cpu().numpy().ravel()
    holdout_true = y_holdout.detach().cpu().numpy().ravel()

    # Calculate accuracy
    train_accuracy = accuracy_score(train_true, train_pred)
    holdout_accuracy = accuracy_score(holdout_true, holdout_pred)

    results.append({
        "Architecture": config["Architecture"],
        "Activation": config["Activation"].title(),
        "Learning Rate": learning_rate,
        "Epochs": epochs,
        "Train Accuracy": train_accuracy,
        "Holdout Accuracy": holdout_accuracy
    })

# Display results, highest holdout accuracy first
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    "Holdout Accuracy", ascending=False
).reset_index(drop=True)

results_df.round(4)

Epoch 0, Cost: 0.4410
Epoch 10, Cost: 0.2874
Epoch 20, Cost: 0.2828
Epoch 30, Cost: 0.2296
Epoch 40, Cost: 0.1462
Epoch 50, Cost: 0.1108
Epoch 60, Cost: 0.0977
Epoch 70, Cost: 0.0888
Epoch 80, Cost: 0.0845
Epoch 90, Cost: 0.0904
Epoch 100, Cost: 0.0751
Epoch 110, Cost: 0.0713
Epoch 120, Cost: 0.0700
Epoch 130, Cost: 0.0691
Epoch 140, Cost: 0.0698
Epoch 0, Cost: 0.3311
Epoch 10, Cost: 0.2925
Epoch 20, Cost: 0.2904
Epoch 30, Cost: 0.2899
Epoch 40, Cost: 0.2893
Epoch 50, Cost: 0.2909
Epoch 60, Cost: 0.2900
Epoch 70, Cost: 0.2890
Epoch 80, Cost: 0.2890
Epoch 90, Cost: 0.2920
Epoch 100, Cost: 0.2915
Epoch 110, Cost: 0.2890
Epoch 120, Cost: 0.2898
Epoch 130, Cost: 0.2890
Epoch 140, Cost: 0.2890
Epoch 0, Cost: 0.3752
Epoch 10, Cost: 0.2914
Epoch 20, Cost: 0.2736
Epoch 30, Cost: 0.2157
Epoch 40, Cost: 0.1315
Epoch 50, Cost: 0.1003
Epoch 60, Cost: 0.0910
Epoch 70, Cost: 0.0838
Epoch 80, Cost: 0.0792
Epoch 90, Cost: 0.0765
Epoch 100, Cost: 0.0752
Epoch 110, Cost: 0.0734
Epoch 120, Cost: 0.0724
E

,Architecture,Activation,Learning Rate,Epochs,Train Accuracy,Holdout Accuracy
0,1 hidden layer (8),Relu,0.01,150,0.9700,0.995
1,"2 hidden layers (32, 32)",Tanh,0.01,150,0.9675,0.990
2,1 hidden layer (32),Relu,0.01,150,0.9750,0.985
3,"2 hidden layers (16, 16)",Relu,0.01,150,0.9750,0.985
4,1 hidden layer (16),Relu,0.01,150,0.9750,0.980
5,1 hidden layer (32),Tanh,0.01,150,0.9600,0.975
6,"2 hidden layers (16, 16)",Tanh,0.01,150,0.9575,0.970
7,"2 hidden layers (32, 32)",Relu,0.01,150,0.9712,0.965
8,"2 hidden layers (8, 8)",Relu,0.01,150,0.9625,0.960
9,1 hidden layer (8),Tanh,0.01,150,0.8675,0.850


Across the board, we can see that increasing the amount of hidden units benefitted performance. However, models with smaller hidden units still performed well. This means that a relatively small network can already learn patterns in this dataset.

Similarly, two layers did not consistently outperform one layer. Some of these models actually performed worse with two layers. For example, the Tanh model with 2 hidden layers of 32 units underperformed the Tanh model with 1 hidden layer of 32 units.

The performance of both was dependent on the architecture. ReLU consistently performed well across all architectures. On the other hand Tanh had two comparatively poor performing models, but it also had good ones. The two worst were the ones with two hidden layers and eight and 32 hidden units. The results suggest that the best activation function is based on architecture.

## 2. Multi-Class Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

---
**NOTE**: This will likely be the most challenging exercise. To help you, here are some pointers:

 - The **output (last) layer** should have as many units as there are classes in your data.  

 - The **activation function of the output layer must be softmax**, not sigmoid. Softmax ensures that all output values are between 0 and 1 and sum to 1, so they can be interpreted as probabilities across the different classes. (*Help*: you may want to look into `torch.nn.functional` to find its implementation...)

 - For a multi-class problem, your target variable `y` should be **one-hot encoded**. For example:  
   - Label = 0 → [1, 0, 0]  
   - Label = 1 → [0, 1, 0]  
   - Label = 2 → [0, 0, 1]  
   You can easily achieve this with `OneHotEncoder` from `sklearn.preprocessing`.  

 - The **predicted class** corresponds to the unit with the highest probability.  
   Example:  
   - `[0.1, 0.3, 0.6] → class 2`  
   - `[0.6, 0.2, 0.2] → class 0`  

 - For this exercise, you need to **implement the categorical cross-entropy loss** (an extension of binary cross-entropy to multiple classes). It is defined as:  

   $$\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

   where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.

---

In [78]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_classes=3,
                           n_clusters_per_class=1,
                           n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [79]:
import torch


class MultiClassMLP:

    def __init__(self, input_dim, hidden_dim, n_classes, hidden_dim2=None, learning_rate=0.01, activation="relu"):
        self.hidden_dim2 = hidden_dim2
        self.activation = activation
        self.learning_rate = learning_rate

        self.W_1 = torch.randn(input_dim, hidden_dim, requires_grad=True)
        self.b_1 = torch.zeros(hidden_dim, requires_grad=True)

        if hidden_dim2 is None:
            self.W_out = torch.randn(
                hidden_dim, n_classes, requires_grad=True
            )
        else:
            self.W_2 = torch.randn(
                hidden_dim, hidden_dim2, requires_grad=True
            )
            self.b_2 = torch.zeros(hidden_dim2, requires_grad=True)
            self.W_out = torch.randn(
                hidden_dim2, n_classes, requires_grad=True
            )

        self.b_out = torch.zeros(n_classes, requires_grad=True)

    def activate(self, z):
        if self.activation == "relu":
            return torch.relu(z)
        else:
            return torch.tanh(z)

    def forward(self, X):
        h = self.activate(X @ self.W_1 + self.b_1)

        if self.hidden_dim2 is not None:
            h = self.activate(h @ self.W_2 + self.b_2)

        out = h @ self.W_out + self.b_out
        return torch.softmax(out, dim=1)

    def cost(self, y_pred, y_true):
        eps = torch.finfo(y_pred.dtype).eps
        y_pred = torch.clamp(y_pred, eps, 1 - eps)

        return -torch.mean(
            torch.sum(y_true * torch.log(y_pred), dim=1)
        )

    def fit(self, X, y, n_epochs=100):
        for epoch in range(n_epochs):
            randperm = torch.randperm(X.shape[0])

            for i in randperm:
                x_batch, y_batch = X[i:i + 1], y[i:i + 1]

                # forward pass
                probs = self.forward(x_batch)
                cost = self.cost(probs, y_batch)

                # backward pass
                cost.backward()

                # update parameters
                with torch.no_grad():
                    self.W_1 -= self.learning_rate * self.W_1.grad
                    self.b_1 -= self.learning_rate * self.b_1.grad
                    self.W_out -= self.learning_rate * self.W_out.grad
                    self.b_out -= self.learning_rate * self.b_out.grad

                    self.W_1.grad.zero_()
                    self.b_1.grad.zero_()
                    self.W_out.grad.zero_()
                    self.b_out.grad.zero_()

                    if self.hidden_dim2 is not None:
                        self.W_2 -= self.learning_rate * self.W_2.grad
                        self.b_2 -= self.learning_rate * self.b_2.grad

                        self.W_2.grad.zero_()
                        self.b_2.grad.zero_()

        return self

    def predict(self, X):
        with torch.no_grad():
            probs = self.forward(X)
            predicted = torch.argmax(probs, dim=1)

        return predicted

In [80]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# split into training and holdout sets (80/20)
X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size=0.2, random_state=42)

# standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_holdout = scaler.transform(X_holdout)

# one-hot encoder
encoder = OneHotEncoder(sparse_output=False)
y_train_onehot = encoder.fit_transform(y_train.reshape(-1, 1))

# convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_holdout = torch.tensor(X_holdout, dtype=torch.float32)
y_train_onehot = torch.tensor(y_train_onehot, dtype=torch.float32)
y_train = torch.tensor(y_train)
y_holdout = torch.tensor(y_holdout)

print("X_train:", X_train.shape, " y_train_onehot:", y_train_onehot.shape)
print("X_holdout:", X_holdout.shape, " y_holdout:", y_holdout.shape)

X_train: torch.Size([800, 2])  y_train_onehot: torch.Size([800, 3])
X_holdout: torch.Size([200, 2])  y_holdout: torch.Size([200])


In [81]:
import pandas as pd
from sklearn.metrics import accuracy_score

torch.manual_seed(1234)

learning_rate = 0.01
epochs = 150
results = []

for activation in ["relu", "tanh"]:
    for n_layers in [1, 2]:
        for units in [8, 16, 32]:
            if n_layers == 1:
                hidden_dim2 = None
            else:
                hidden_dim2 = units

            # create and train a fresh model for each configuration
            model = MultiClassMLP(input_dim=2, hidden_dim=units, n_classes=3, hidden_dim2=hidden_dim2, learning_rate=learning_rate, activation=activation)
            model.fit(X_train, y_train_onehot, n_epochs=epochs)

            # accuracy on both the training and holdout sets
            results.append({
                "Activation": activation,
                "Hidden layers": n_layers,
                "Units per layer": units,
                "Train accuracy": accuracy_score(y_train, model.predict(X_train)),
                "Holdout accuracy": accuracy_score(y_holdout, model.predict(X_holdout))
            })

results_df = pd.DataFrame(results)
results_df

,Activation,Hidden layers,Units per layer,Train accuracy,Holdout accuracy
0,relu,1,8,0.82250,0.805
1,relu,1,16,0.81625,0.800
2,relu,1,32,0.82375,0.790
3,relu,2,8,0.81875,0.800
4,relu,2,16,0.82375,0.810
5,relu,2,32,0.82875,0.810
6,tanh,1,8,0.82750,0.795
7,tanh,1,16,0.81750,0.800
8,tanh,1,32,0.81875,0.805
9,tanh,2,8,0.82875,0.805


### Reflection and discussion (multi-class)

The 12 configurations all got similar results, about 81.6–84.3% accuracy on the training set and 79–82.5% on the holdout set. This is close to the best we can expect for this dataset because flip_y=0.15 randomly reassigns 15% of the labels, so roughly 10% of the points have the wrong label, and no model can predict them.

Different Hidden Units: With one hidden layer, going from 8 to 16 to 32 units changes holdout accuracy by at most 1.5 percentage points (79–80.5%). Fairly simple boundaries separate the classes, so 8 units already give the model enough capacity, and adding more units does not help.

ReLU vs. Tanh: The activation function creates a little difference. ReLU models score 79–82.5% on the holdout set and Tanh models 79.5–82%. Tanh is slightly more consistent, likely because its output is smooth and centered at zero, while ReLU creates piecewise-linear boundaries. For this data, both activations are flexible enough, and the label noise limits performance more than the choice of activation.

Different Hidden Layer: Adding a second layer slightly increases the highest training accuracy, from 82.8% to 84.3%, and yields a few of the best holdout scores (82–82.5%). However, the improvement is small and not consistent (for example, ReLU with 8 or 16 units does no better than one layer). A second layer adds flexibility. However, on a simple, noisy dataset like this, it mostly fits the training data a bit better rather than generalizing better. The gap between training and holdout accuracy stays small for all models.

## 3: Regression Task (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [82]:
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

In [83]:
# Step 1: Implementation

class RegressionMLP:
  def __init__(
        self,
        input_dim,
        hidden_dim,
        hidden_dim2=None,
        learning_rate=0.1,
        activation="relu"
    ):
        self.learning_rate = learning_rate
        self.activation = activation
        self.hidden_dim2 = hidden_dim2

        self.W1 = torch.randn(input_dim, hidden_dim) * 0.1
        self.b1 = torch.zeros(hidden_dim)

        if hidden_dim2 is None:
            self.W_out = torch.randn(hidden_dim, 1) * 0.1
            self.params = [self.W1, self.b1]
        else:
            self.W2 = torch.randn(hidden_dim, hidden_dim2) * 0.1
            self.b2 = torch.zeros(hidden_dim2)
            self.W_out = torch.randn(hidden_dim2, 1) * 0.1
            self.params = [self.W1, self.b1, self.W2, self.b2]

        self.b_out = torch.zeros(1)
        self.params += [self.W_out, self.b_out]

        for p in self.params:
            p.requires_grad = True

  def forward(self, X):
      self.h1 = X @ self.W1 + self.b1
      if self.activation == "relu":
          self.a1 = torch.relu(self.h1)
      else:
          self.a1 = torch.tanh(self.h1)

      if self.hidden_dim2 is not None:
          self.h2 = self.a1 @ self.W2 + self.b2
          if self.activation == "relu":
              self.a2 = torch.relu(self.h2)
          else:
              self.a2 = torch.tanh(self.h2)

          hidden_output = self.a2
      else:
          hidden_output = self.a1

      output = hidden_output @ self.W_out + self.b_out
      return output

  def cost(self, y_pred, y):
      return torch.mean((y_pred - y) ** 2).mean()

  def fit (self, X, y, X_hold, y_hold, epochs=200):
      self.train_costs = []
      self.hold_costs = []
      n = len(X)

      for epoch in range (epochs):
        order = torch.randperm(n)

        for i in range (0, n, 32):
          batch_idx = order[i:i+32]
          loss = self.cost(self.forward(X[batch_idx]), y[batch_idx])
          loss.backward()

          with torch.no_grad():
            for p in self.params:
              p -= self.learning_rate * p.grad
              p.grad.zero_()

        with torch.no_grad():
          self.train_costs.append(self.cost(self.forward(X), y).item())
          self.hold_costs.append(self.cost(self.forward(X_hold), y_hold).item())
  def predict(self, X):
      with torch.no_grad():
        return self.forward(X)



In [84]:
# Step 2: Data Preperation
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

X, y = make_regression(n_samples=1000, n_features=2, n_informative=2,
                       random_state=1234, noise=75)

X_train, X_hold, y_train, y_hold = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_hold = scaler.transform(X_hold)

y_scaler = StandardScaler()
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1))
y_hold_s = y_scaler.transform(y_hold.reshape(-1, 1))

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_hold_t = torch.tensor(X_hold, dtype=torch.float32)
y_train_t = torch.tensor(y_train_s, dtype=torch.float32)
y_hold_t = torch.tensor(y_hold_s, dtype=torch.float32)

print("training set:", X_train_t.shape)
print("holdout set:", X_hold_t.shape)


training set: torch.Size([800, 2])
holdout set: torch.Size([200, 2])


In [85]:
# Step 3: Model Training and Evaluation

def mse_original_units(model, X_t, y_true):
    pred = model.predict(X_t).numpy()
    pred = y_scaler.inverse_transform(pred).flatten()
    return ((pred - y_true) ** 2).mean()

def train_and_score(n_hidden, n_layers, activation):
  if n_layers == 2:
        hidden_dim2 = n_hidden
  else:
        hidden_dim2 = None
  model = RegressionMLP(
        input_dim=X_train_t.shape[1],
        hidden_dim=n_hidden,
        hidden_dim2=hidden_dim2,
        learning_rate=0.01,
        activation=activation
  )
  model.fit(X_train_t, y_train_t, X_hold_t, y_hold_t, epochs=200)
  row = {"activation": activation,
         "layers": n_layers,
         "units": n_hidden,
         "train MSE": round(mse_original_units(model, X_train_t, y_train)),
         "holdout MSE": round(mse_original_units(model, X_hold_t, y_hold))}
  return model, row

In [86]:
# Experiment 1: 1 hidden layer, different numbers of units

rows_A = []
for activation in ["relu", "tanh"]:
    for units in [1, 2, 4, 8, 16, 32, 64, 128]:
        model, row = train_and_score(units, 1, activation)
        rows_A.append(row)

table_A = pd.DataFrame(rows_A)
table_A

,activation,layers,units,train MSE,holdout MSE
0,relu,1,1,5510,5503
1,relu,1,2,5439,5561
2,relu,1,4,5444,5553
3,relu,1,8,5435,5625
4,relu,1,16,5396,5538
5,relu,1,32,5393,5548
6,relu,1,64,5382,5529
7,relu,1,128,5374,5496
8,tanh,1,1,5549,5538
9,tanh,1,2,5507,5545


In [87]:
# Experiment 2: Same amount of units, comparing the number of hidden layers

rows_B = []
models = {}                                   # keep these for the plots below
for activation in ["relu", "tanh"]:
    for units in [4, 16, 64]:
        for layers in [1, 2]:
            model, row = train_and_score(units, layers, activation)
            rows_B.append(row)
            models[(activation, layers, units)] = model

table_B = pd.DataFrame(rows_B)
table_B

,activation,layers,units,train MSE,holdout MSE
0,relu,1,4,5480,5464
1,relu,2,4,5415,5517
2,relu,1,16,5387,5525
3,relu,2,16,5412,5527
4,relu,1,64,5380,5476
5,relu,2,64,5352,5491
6,tanh,1,4,5507,5525
7,tanh,2,4,5526,5544
8,tanh,1,16,5496,5516
9,tanh,2,16,5494,5527


In [88]:
# Baseline: Typical Linear Regression

from sklearn.linear_model import LinearRegression

lin = LinearRegression().fit(X_train, y_train)
print("train MSE:  ", round(((lin.predict(X_train) - y_train) ** 2).mean()))
print("holdout MSE:", round(((lin.predict(X_hold) - y_hold) ** 2).mean()))

train MSE:   5495
holdout MSE: 5491


### Reflection and discussion (Regression)

In ReLU, the training MSE steadily decreases as the number of units goes up, while the holdout MSE hovers between 5480 and 5670. Since the extra units improve the fit of the model in the training set but does not do the same to the holdout set, that is a sign of some overfitting.

With Tanh, there's little to no change at all once the number of units exceeds 2. Because the true relationship is linear and there are only 2 input features, a small network is already enough. Extra units mostly add the ability to fit noise.

With ReLU, adding a second layer gave the lowest amount of training MSE of all models, at 64 units. The only area where adding a second layer can make the model perform worse with 4 units. So, a narrow second layer can lose information, and a deeper network is harder to train.

With Tanh, adding a second layer made almost no difference, as the differences in MSE between 1 and 2 layers were all under 40, and for the holdout it was either the same or slightly worse. The differences are small compared with the noise in the data, so a second layer brings no real benefit for this simple, linear problem. Depth is more useful when the true relationship is complex.

ReLU in general reaches a lower training MSE, particularly for larger networks, but it's susceptibility to fitting noise makes its results vary more from one configuration to the next. Meanwhile, tanh barely has any variation at all and showed very little signs of overfitting, so it acts very much like linear regression.


## 4. Discussion (5 points)

You created a separate class for each task and likely repeated much of the same code across implementations.

Discuss within your group how could you have leveraged inheritance to make your code more reusable and avoid duplication. Provide examples. Be specific.

We all duplicated the __init__ section with W_1, b_1, W_2, b_2, W_out, and b_out, the hidden layer part of forward, and the SGD loop in fit. The only things that realistically needed to differ were the output units, output activation function, cost function, and predict function. Because we all had three variations of the same code, it ended up creating different names for the same things, different initializations, and different gradient methods. With inheritance, all of this clutter can go away by living in one place.

A solution to this would be making a parent class called something like "BaseMLP", that holds everything that needs to be shared and calls hook methods that each individual child class can fill in with just a few lines. This would mean less code, fewer bugs, and more consistency amongst the codeYOUR TEXT HERE

## 5. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

YOUR TEXT HERE